In [58]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pymongo import MongoClient, UpdateOne

# ==========================================
# CONFIGURACIÓN
# ==========================================
BASE_URL = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/{}"
MAX_PAGINAS = 6   # <- subir a 6 para intentar superar 50 registros

NOMBRE_BD = "Inmobiliaria"
NOMBRE_COLECCION = "chilepropiedades_laserena_raw"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# ==========================================
# FUNCIONES AUXILIARES
# ==========================================
def texto_limpio(x):
    if not x:
        return None
    return " ".join(x.get_text(" ", strip=True).split())

def texto_limpio_str(x):
    if not x:
        return None
    return " ".join(str(x).split())

def obtener_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def normalizar_url(href):
    if not href:
        return None
    if href.startswith("http"):
        return href
    if href.startswith("/"):
        return "https://chilepropiedades.cl" + href
    return "https://chilepropiedades.cl/" + href

# ==========================================
# SCRAPING
# ==========================================
publicaciones = []
vistos = set()

for pagina in range(1, MAX_PAGINAS + 1):
    url = BASE_URL.format(pagina)
    print(f"\n📄 Procesando listado: {url}")

    try:
        soup = obtener_soup(url)
    except Exception as e:
        print(f"⚠️ Error al cargar listado {url}: {e}")
        continue

    links = soup.find_all("a", href=True)
    candidatos = []

    for a in links:
        href = a.get("href")
        href_full = normalizar_url(href)

        if not href_full:
            continue

        if "/ver-publicacion/" in href_full and href_full not in vistos:
            vistos.add(href_full)
            candidatos.append(href_full)

    print(f"🔎 Publicaciones detectadas: {len(candidatos)}")

    for enlace in candidatos:
        try:
            detalle = obtener_soup(enlace)
            texto_total = texto_limpio_str(detalle.get_text(" ", strip=True))

            titulo = None
            h1 = detalle.find("h1")
            if h1:
                titulo = texto_limpio(h1)

            if not titulo:
                title_tag = detalle.find("title")
                if title_tag:
                    titulo = texto_limpio(title_tag)

            precio_texto = None
            m_precio = re.search(r"\$\s*[\d\.\,]+", texto_total)
            if m_precio:
                precio_texto = m_precio.group(0)

            precio_num = None
            if precio_texto:
                nums = re.sub(r"[^\d]", "", precio_texto)
                if nums:
                    precio_num = int(nums)

            ubicacion = "La Serena" if "La Serena" in texto_total else None

            # Extraer mejor desde el texto completo
            habitaciones = None
            m_hab = re.search(r"(\d+)\s*Baño:", texto_total)
            if m_hab:
                habitaciones = m_hab.group(1)

            banos = None
            m_ban = re.search(r"Baño:\s*(\d+)", texto_total)
            if m_ban:
                banos = m_ban.group(1)

            terreno = None
            m_ter = re.search(r"Superficie Total:\s*(\d+)\s*m²", texto_total)
            if m_ter:
                terreno = m_ter.group(1)

            estacionamientos = None
            m_est = re.search(r"Estacionamientos:\s*(\d+)", texto_total)
            if m_est:
                estacionamientos = m_est.group(1)

            keywords = [
                "Amoblado", "Terraza", "Lavandería", "Lavanderia", "Gimnasio",
                "Bodega", "Piscina", "Logia", "Conserjería", "Conserjeria",
                "Ascensor", "Suite", "Quincho", "Seguridad", "Balcón", "Balcon"
            ]

            atributos_detectados = []
            for kw in keywords:
                if re.search(rf"\b{re.escape(kw)}\b", texto_total, re.IGNORECASE):
                    atributos_detectados.append(kw)

            atributos_detectados = list(dict.fromkeys(atributos_detectados))

            descripcion = None
            m_desc = re.search(r"Descripción\s+(.*?)(Mapa -|Código aviso:|$)", texto_total, re.IGNORECASE)
            if m_desc:
                descripcion = m_desc.group(1).strip()

            if not descripcion:
                descripcion = texto_total[:2500]

            doc = {
                "titulo": titulo,
                "precio_texto": precio_texto,
                "precio_num": precio_num,
                "ubicacion": ubicacion,
                "habitaciones": habitaciones,
                "banos": banos,
                "terreno": terreno,
                "estacionamientos": estacionamientos,
                "atributos_detectados": atributos_detectados,
                "descripcion": descripcion,
                "enlace": enlace,
                "fuente": "ChilePropiedades",
                "tipo_operacion": "Arriendo mensual",
                "tipo_propiedad": "Departamento",
                "ciudad_objetivo": "La Serena",
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            publicaciones.append(doc)

        except Exception as e:
            print(f"⚠️ Error en detalle {enlace}: {e}")
            continue

# ==========================================
# ELIMINAR DUPLICADOS DEL CORRIDO
# ==========================================
finales = []
enlaces_finales = set()

for p in publicaciones:
    enlace = p.get("enlace")
    if enlace and enlace not in enlaces_finales:
        enlaces_finales.add(enlace)
        finales.append(p)

print(f"\n✅ Total final sin duplicados: {len(finales)}")

# ==========================================
# CSV
# ==========================================
df_csv = pd.DataFrame(finales)
df_csv.to_csv("chilepropiedades_laserena_raw.csv", index=False, encoding="utf-8-sig")
print("💾 CSV generado")

# ==========================================
# MONGODB SIN DUPLICADOS
# ==========================================
client = MongoClient("mongodb://bigdata_mongodb:27017/", serverSelectionTimeoutMS=5000)
db = client[NOMBRE_BD]
coleccion = db[NOMBRE_COLECCION]

operaciones = []
for doc in finales:
    operaciones.append(
        UpdateOne(
            {"enlace": doc["enlace"]},
            {"$set": doc},
            upsert=True
        )
    )

if operaciones:
    resultado = coleccion.bulk_write(operaciones)
    print("✅ Datos guardados/actualizados en MongoDB")
else:
    print("⚠️ No hubo datos para guardar")


📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/1
🔎 Publicaciones detectadas: 10

📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/2
🔎 Publicaciones detectadas: 10

📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/3
🔎 Publicaciones detectadas: 10

📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/4
🔎 Publicaciones detectadas: 10

📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/5
🔎 Publicaciones detectadas: 10

📄 Procesando listado: https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/la-serena/6
🔎 Publicaciones detectadas: 10

✅ Total final sin duplicados: 60
💾 CSV generado
✅ Datos guardados/actualizados en MongoDB


In [62]:
from pyspark.sql import functions as F

df_limpio = (
    df_raw
    .withColumn("precio", F.col("precio_num").cast("double"))
    .withColumn("habitaciones_num", F.col("habitaciones").cast("int"))
    .withColumn("banos_num", F.col("banos").cast("int"))
    .withColumn("terreno_m2", F.col("terreno").cast("int"))
    .withColumn("estacionamientos_num", F.col("estacionamientos").cast("int"))
    .filter((F.col("precio").isNotNull()) & (F.col("precio") > 0))
    .dropDuplicates(["enlace"])
)

print("✅ Total limpio:", df_limpio.count())
df_limpio.show(30, truncate=False)

✅ Total limpio: 30
+---------------------------------------------------------------+------------+----------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [64]:
df_analisis = df_limpio.withColumn(
    "precio_m2",
    F.when(
        (F.col("terreno_m2").isNotNull()) & (F.col("terreno_m2") > 0),
        F.round(F.col("precio") / F.col("terreno_m2"), 2)
    ).otherwise(None)
)

df_analisis.select(
    "titulo",
    "precio",
    "terreno_m2",
    "precio_m2"
).show(30, truncate=False)

+---------------------------------------------------------------+---------+----------+---------+
|titulo                                                         |precio   |terreno_m2|precio_m2|
+---------------------------------------------------------------+---------+----------+---------+
|La Serena, Amoblado (158821)                                   |560000.0 |NULL      |NULL     |
|La Serena, Amoblado – (159096)                                 |460000.0 |NULL      |NULL     |
|La Serena, Arriendo 2Destac. | 50% OFF mes 1 (158795)          |700000.0 |NULL      |NULL     |
|La Serena, Arriendo año corrido barrio universitario la serena |680000.0 |NULL      |NULL     |
|La Serena, Arriendo dpto peñuelas coquimbo (159124)            |650000.0 |NULL      |NULL     |
|La Serena, Arriendo hermoso departamento en la serena (158769) |850000.0 |NULL      |NULL     |
|La Serena, Av. Cruz del Molino 325                             |400000.0 |NULL      |NULL     |
|La Serena, Av. del Mar 560   

In [65]:
df_analisis.groupBy("habitaciones_num").agg(
    F.count("*").alias("cantidad"),
    F.round(F.avg("precio"), 0).alias("precio_promedio"),
    F.round(F.avg("terreno_m2"), 0).alias("m2_promedio"),
    F.round(F.avg("precio_m2"), 2).alias("precio_m2_promedio")
).orderBy("habitaciones_num").show()

+----------------+--------+---------------+-----------+------------------+
|habitaciones_num|cantidad|precio_promedio|m2_promedio|precio_m2_promedio|
+----------------+--------+---------------+-----------+------------------+
|            NULL|      30|       620000.0|       NULL|              NULL|
+----------------+--------+---------------+-----------+------------------+

